1. Loading Data

In [1]:
import pandas as pd

movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

df = ratings.merge(movies, on='movieId')

df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


2. Creating User-Movie Matrix

In [2]:
user_movie_matrix = df.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

user_movie_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


3. Filling Missing Values

In [3]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

4. Compute Similarity (User-Based)

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix_filled)

print("User similarity matrix shape:", user_similarity.shape)

User similarity matrix shape: (610, 610)


5. Converting to DaraFrame

In [5]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


6. Finding similar Users

In [6]:
def get_similar_users(user_id, top_n=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    similar_users = similar_users.drop(user_id)
    return similar_users.head(top_n)

get_similar_users(1)

userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64

7. Recommend Movies Based on Similar Users

In [7]:
def recommend_for_user(user_id, top_n=10):
    
    similar_users = get_similar_users(user_id, top_n=5)
    similar_users_ids = similar_users.index
    
    similar_users_movies = user_movie_matrix.loc[similar_users_ids]
    
    mean_ratings = similar_users_movies.mean().sort_values(ascending=False)
    
    user_movies = user_movie_matrix.loc[user_id].dropna().index
    
    recommendations = mean_ratings.drop(user_movies, errors='ignore')
    
    return recommendations.head(top_n)

recommend_for_user(1)

title
Flash Gordon (1980)                                                            5.0
General, The (1926)                                                            5.0
Alien Nation (1988)                                                            5.0
Godfather, The (1972)                                                          5.0
Waiting for Guffman (1996)                                                     5.0
Wallace & Gromit: The Best of Aardman Animation (1996)                         5.0
High Plains Drifter (1973)                                                     5.0
Harold and Maude (1971)                                                        5.0
Hard-Boiled (Lat sau san taam) (1992)                                          5.0
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)    5.0
dtype: float64

8. Transpose Matrix

In [8]:
item_movie_matrix = user_movie_matrix.T
item_movie_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


9. Improve Quality (Shortlist Booster)

In [9]:
# Add Minimum Rating Filter
movie_counts = df.groupby('title')['rating'].count()
popular_movies = movie_counts[movie_counts > 50].index

# Apply Filter BEFORE similarity
item_movie_matrix = item_movie_matrix.loc[popular_movies]

10. Compute Item Similarity

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(item_movie_matrix.fillna(0))

print(item_similarity.shape)

(437, 437)


11. Convert to DataFrame

In [11]:
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_movie_matrix.index,
    columns=item_movie_matrix.index
)

item_similarity_df.head()

title,10 Things I Hate About You (1999),12 Angry Men (1957),2001: A Space Odyssey (1968),28 Days Later (2002),300 (2007),"40-Year-Old Virgin, The (2005)",A.I. Artificial Intelligence (2001),"Abyss, The (1989)",Ace Ventura: Pet Detective (1994),Ace Ventura: When Nature Calls (1995),...,Wild Wild West (1999),Willy Wonka & the Chocolate Factory (1971),"Wizard of Oz, The (1939)","Wolf of Wall Street, The (2013)",X-Men (2000),X-Men: The Last Stand (2006),X2: X-Men United (2003),Young Frankenstein (1974),Zombieland (2009),Zoolander (2001)
title,,,,,,,,,,,,,,,,,,,,,
10 Things I Hate About You (1999),1.000000,0.095586,0.191393,0.170720,0.351032,0.350903,0.221459,0.155244,0.229660,0.213071,...,0.332358,0.280982,0.289917,0.133364,0.285599,0.215151,0.217504,0.224106,0.227712,0.340590
12 Angry Men (1957),0.095586,1.000000,0.291756,0.217353,0.261876,0.236899,0.209119,0.162227,0.198630,0.186848,...,0.179743,0.230947,0.324110,0.188794,0.237337,0.254711,0.235636,0.220096,0.217742,0.196745
2001: A Space Odyssey (1968),0.191393,0.291756,1.000000,0.398424,0.339076,0.317919,0.460043,0.490033,0.271807,0.230101,...,0.305921,0.301007,0.389963,0.191644,0.390909,0.340882,0.333269,0.403752,0.288656,0.246335
28 Days Later (2002),0.170720,0.217353,0.398424,1.000000,0.423250,0.387243,0.437794,0.206805,0.206628,0.194713,...,0.229422,0.267909,0.266093,0.228052,0.348975,0.403226,0.410175,0.218362,0.436973,0.296377
300 (2007),0.351032,0.261876,0.339076,0.423250,1.000000,0.611255,0.394823,0.228963,0.361877,0.319655,...,0.440699,0.292031,0.314623,0.295794,0.471374,0.544290,0.475067,0.212877,0.480951,0.464538


12. Movie-to-Movie Recommendation

In [12]:
def recommend_similar_movies(movie_title, top_n=10):
    
    movie_title = movie_title.lower().strip()
    
    matches = [m for m in item_similarity_df.columns if movie_title in m.lower()]
    
    if not matches:
        return "Movie not found"
    
    matched_movie = matches[0]
    
    similar_scores = item_similarity_df[matched_movie].sort_values(ascending=False)
    
    similar_scores = similar_scores.drop(matched_movie)
    
    return similar_scores.head(top_n)

13. Testing

In [13]:
recommend_similar_movies('Toy Story')

title
Toy Story 2 (1999)                                   0.572601
Jurassic Park (1993)                                 0.565637
Independence Day (a.k.a. ID4) (1996)                 0.564262
Star Wars: Episode IV - A New Hope (1977)            0.557388
Forrest Gump (1994)                                  0.547096
Lion King, The (1994)                                0.541145
Star Wars: Episode VI - Return of the Jedi (1983)    0.541089
Mission: Impossible (1996)                           0.538913
Groundhog Day (1993)                                 0.534169
Back to the Future (1985)                            0.530381
Name: Toy Story (1995), dtype: float64

14. Saving Model

In [14]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/item_similarity.pkl', 'wb') as f:
    pickle.dump(item_similarity_df, f)